# 토큰 단위 BiLSTM 시퀀스 라벨링 — 스팬 추출 (실제 캐글 제출)

**연습 기법** (IOAI 2025 GAITE **Word Segmentation** 과 동일한 *시퀀스 라벨링*): 텍스트의 **각 토큰에 0/1 을 다는**
방식으로 부분을 찾아낸다. Word Segmentation 은 글자마다 `1=단어 끝` 을 예측했고(20번 노트북), 여기선 **토큰마다
`1=정답 스팬에 포함`** 을 예측한다 — 둘 다 **임베딩 + BiLSTM → 토큰별 이진분류**.

**대회**: [Tweet Sentiment Extraction](https://www.kaggle.com/c/tweet-sentiment-extraction) — 트윗과 감성
(positive/negative/neutral)이 주어지면, **그 감성을 드러내는 부분 문자열(selected_text)** 을 뽑는 **스팬 추출**
과제. 20번(구성한 과제, 캐글 LB 무관)과 달리 **실제 캐글 대회**라 `textID,selected_text` 형식으로 **진짜 제출**한다.
데이터도 가볍다(train 3.5MB). 지표는 단어 단위 **Jaccard**.

**핵심 흐름 & 배울 점**:
1. **토큰별 라벨 만들기** — 정답 스팬의 문자 범위와 겹치는 토큰을 1 로.
2. **단어 임베딩 + 감성 임베딩 + BiLSTM** → 토큰별 포함 여부 예측.
3. **스팬 복원** — 예측 1 토큰의 처음~끝을 부분문자열로(연속 스팬). `neutral` 은 보통 전체가 정답이라 전체 텍스트.
4. **Jaccard** 로 평가(전체-텍스트 baseline 과 비교) → 실제 `submission.csv` 제출.

> ⚙️ GPU 권장(작은 BiLSTM, T4 ~1분). 데이터가 작아 CPU 도 가능.
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.
> 🔑 **첫 실행 시 대회 Rules 수락 필요**: 403 이 나면 [대회 페이지](https://www.kaggle.com/c/tweet-sentiment-extraction/rules)
> 에서 **"I Understand and Accept"** 1회 클릭 후 다시 실행하세요(자동화 불가).

## 0. 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle 에서 데이터 다운로드
403 이 나면 대회 Rules 를 한 번 수락해야 합니다(위 안내 참조).

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
try:
    api.competition_download_files("tweet-sentiment-extraction", path=WORK_DIR, quiet=False)
    for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
        with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
        os.remove(zp)
    print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])
except Exception as e:
    print("⚠️ 다운로드 실패:", repr(e)[:200])
    print("→ 403 이면 https://www.kaggle.com/c/tweet-sentiment-extraction/rules 에서 'I Understand and Accept' 1회 후 재실행")

## 2. 토큰별 라벨 만들기 — 정답 스팬에 포함되면 1
`selected_text` 의 **문자 범위**를 찾고, 그 범위와 겹치는 텍스트 토큰을 1 로 라벨링한다. (Word Segmentation 의
글자별 라벨과 같은 *토큰별 시퀀스 라벨링*)

In [ ]:
import re, numpy as np, pandas as pd
train = pd.read_csv(os.path.join(WORK_DIR, "train.csv")).dropna(subset=["text", "selected_text"]).reset_index(drop=True)
test  = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
SENT = {"positive": 0, "negative": 1, "neutral": 2}

def token_offsets(text):
    return [(m.group(), m.start(), m.end()) for m in re.finditer(r"\S+", str(text))]

def token_labels(text, selected):
    i = str(text).find(str(selected)); s, e = (i, i + len(str(selected)))
    labs = []
    for tok, a, b in token_offsets(text):
        labs.append(1 if (i >= 0 and a < e and b > s) else 0)
    return labs

examples = []
for _, r in train.iterrows():
    toks = token_offsets(r["text"])
    if toks:
        examples.append((toks, token_labels(r["text"], r["selected_text"]), SENT[r["sentiment"]],
                         r["text"], r["selected_text"]))

vocab = {"<pad>": 0, "<unk>": 1}
for toks, _, _, _, _ in examples:
    for w, _, _ in toks: vocab.setdefault(w.lower(), len(vocab))
enc = lambda toks: [vocab.get(w.lower(), 1) for w, _, _ in toks]
print("examples:", len(examples), "| vocab:", len(vocab))
print("예시:", examples[0][3], "||", examples[0][4], "|| labels", examples[0][1])

## 3. 모델 — 단어 임베딩 + 감성 임베딩 + BiLSTM
각 토큰 임베딩에 문장 **감성 임베딩**을 붙여 BiLSTM 에 넣고, 토큰마다 포함 로짓 1개를 출력한다.

In [ ]:
import torch, torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SpanTagger(nn.Module):
    def __init__(self, vocab_size, emb=64, sent_emb=16, hid=64):
        super().__init__()
        self.word = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.sent = nn.Embedding(3, sent_emb)
        self.lstm = nn.LSTM(emb + sent_emb, hid, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hid * 2, 1)
    def forward(self, x, sent):
        e = torch.cat([self.word(x), self.sent(sent)[:, None, :].expand(-1, x.shape[1], -1)], -1)
        return self.fc(self.lstm(e)[0]).squeeze(-1)

print("SpanTagger 파라미터:", sum(p.numel() for p in SpanTagger(len(vocab)).parameters()))

## 4. 학습 & 평가 — 마스킹 BCE, **Jaccard** (vs 전체-텍스트 baseline)
추론은 예측 1 토큰의 **처음~끝**을 부분문자열로 만든다(연속 스팬). `neutral` 트윗은 보통 전체가 정답이라
전체 텍스트를 그대로 쓴다(잘 알려진 규칙).

In [ ]:
rng = np.random.RandomState(0); idx = rng.permutation(len(examples)); cut = int(len(examples) * 0.9)
tr_data = [examples[i] for i in idx[:cut]]; va_data = [examples[i] for i in idx[cut:]]

def batchify(data, bs=64, shuffle=False):
    order = np.random.permutation(len(data)) if shuffle else np.arange(len(data))
    for i in range(0, len(order), bs):
        chunk = [data[j] for j in order[i:i+bs]]; m = max(len(t) for t, _, _, _, _ in chunk)
        X = np.zeros((len(chunk), m), "int64"); Y = np.zeros((len(chunk), m), "float32")
        M = np.zeros((len(chunk), m), "float32"); S = np.zeros(len(chunk), "int64")
        for j, (toks, labs, se, _, _) in enumerate(chunk):
            X[j, :len(toks)] = enc(toks); Y[j, :len(labs)] = labs; M[j, :len(toks)] = 1; S[j] = se
        yield (torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(M), torch.from_numpy(S), chunk)

def jaccard(a, b):
    a, b = set(str(a).lower().split()), set(str(b).lower().split())
    return len(a & b) / len(a | b) if (a | b) else 0.0

def decode(text, toks, pred, sentiment):
    if sentiment == SENT["neutral"]: return text
    on = [k for k in range(len(toks)) if pred[k]]
    if not on: return text
    return text[toks[on[0]][1]: toks[on[-1]][2]]

torch.manual_seed(0)
model = SpanTagger(len(vocab)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3); bce = nn.BCEWithLogitsLoss(reduction="none")
EPOCHS = 8
for epoch in range(1, EPOCHS + 1):
    model.train()
    for X, Y, M, S, _ in batchify(tr_data, shuffle=True):
        X, Y, M, S = X.to(device), Y.to(device), M.to(device), S.to(device)
        opt.zero_grad(); loss = (bce(model(X, S), Y) * M).sum() / M.sum(); loss.backward(); opt.step()
    model.eval(); js, jb = [], []
    with torch.no_grad():
        for X, Y, M, S, chunk in batchify(va_data, bs=128):
            p = (torch.sigmoid(model(X.to(device), S.to(device))) > 0.5).cpu().numpy()
            for j, (toks, labs, se, text, sel) in enumerate(chunk):
                js.append(jaccard(decode(text, toks, p[j], se), sel)); jb.append(jaccard(text, sel))
    print(f"epoch {epoch} | val Jaccard = {np.mean(js):.4f}  (전체-텍스트 baseline = {np.mean(jb):.4f})")
print("\n→ 토큰별 BiLSTM 시퀀스 라벨링으로 스팬 추출 — Word Segmentation 과 같은 기법, 실제 캐글 제출.")

## 5. 캐글 제출 파일 생성 (`textID,selected_text`)

In [ ]:
model.eval(); preds = []
for _, r in test.iterrows():
    text = str(r["text"]); toks = token_offsets(text); se = SENT[r["sentiment"]]
    if not toks or se == SENT["neutral"]:
        preds.append(text); continue
    x = torch.tensor([enc(toks)], dtype=torch.long, device=device)
    s = torch.tensor([se], dtype=torch.long, device=device)
    with torch.no_grad():
        p = (torch.sigmoid(model(x, s))[0] > 0.5).cpu().numpy()
    preds.append(decode(text, toks, p, se))

submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"textID": test["textID"], "selected_text": preds}).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(preds))
pd.read_csv(submission_path).head()

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [Tweet Sentiment Extraction](https://www.kaggle.com/c/tweet-sentiment-extraction/submit) 에 제출하세요.

**기법 요약**: Word Segmentation 과 같은 **토큰 단위 시퀀스 라벨링**(임베딩+BiLSTM, 토큰별 0/1)을 **실제 캐글 스팬
추출** 대회에 적용 — 정답 스팬 포함 토큰을 예측해 부분문자열을 복원, **Jaccard** 채점, `selected_text` 제출.
더 끌어올리려면: 사전학습 임베딩/Transformer(RoBERTa) 인코더, 시작·끝 포인터(span pointer) 헤드, 토큰화 정교화
(공백·문장부호), `neutral` 외 임계값 튜닝. (20번은 글자 단위 분할의 *직접 analog*, 21번은 같은 기법의 *실제 제출* 버전.)